# Multi-Class Land Cover Classification - Grid Search (5 Classes)

This notebook performs hyperparameter tuning for a 5-class land cover classification on Jabalpur:
- 0: Forest
- 1: Water
- 2: Buildings
- 3: Soil
- 4: Agriculture

It uses 10 spectral features and performs a 5-fold GridSearchCV across RF and SVM.


In [26]:
import os
import ee
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

load_dotenv()
ee.Initialize(project=os.getenv('EE_PROJECT_ID'))
print("EE initialized.")

EE initialized.


In [27]:
jabalpur = ee.FeatureCollection('FAO/GAUL/2025/level2').filter(
    ee.Filter.And(
        ee.Filter.eq('GAUL2_NAME', 'Jabalpur'),
        ee.Filter.eq('GAUL0_NAME', 'India')
    )
).geometry()

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud  = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask).divide(10000)

print("Loading Earth Engine data")
dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterDate('2025-03-01', '2025-04-30')
      .filterBounds(jabalpur)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2_clouds)
)
composite = dataset.median().clip(jabalpur)
print("Sentinel-2 median composite created")


Loading Earth Engine data
Sentinel-2 median composite created


In [28]:
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')

savi = composite.expression(
    '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
    {'NIR': composite.select('B8'), 'RED': composite.select('B4')}
).rename('SAVI')

composite = composite.addBands([ndvi, ndwi, ndbi, savi])

bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'SAVI']
print(f"Feature bands: {bands}")

Feature bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'SAVI']


In [29]:
water       = ee.FeatureCollection('users/cosypix/jabalpur_water_points').map(lambda f: f.set('label', 1))
forest      = ee.FeatureCollection('users/cosypix/jabalpur_forest_points').map(lambda f: f.set('label', 0))
soil        = ee.FeatureCollection('users/cosypix/jabalpur_barren_points').map(lambda f: f.set('label', 3))
buildings   = ee.FeatureCollection('users/cosypix/jabalpur_urban_points').map(lambda f: f.set('label', 2))
agriculture = ee.FeatureCollection('users/cosypix/jabalpur_agriculture_points').map(lambda f: f.set('label', 4))

all_points = water.merge(forest).merge(soil).merge(buildings).merge(agriculture)

print("Sampling feature regions...")
training = (
    composite.select(bands)
             .sampleRegions(
                 collection=all_points,
                 properties=['label'],
                 scale=10
             )
             .filter(ee.Filter.notNull(bands + ['label']))
)

data_list = training.getInfo()['features']
print(f"Extraction complete 
{len(data_list)} points found.")


Sampling feature regions...
Extraction complete 
5002 points found.


In [30]:
df = pd.DataFrame([feat['properties'] for feat in data_list])
X  = df[bands]
y  = df['label']

print(f"Dataset shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts().sort_index()}")

Dataset shape: (4000, 10)
Class distribution:
label
0    1002
1    1000
2     998
3    1000
Name: count, dtype: int64


In [31]:
rf_grid = {
    'n_estimators':     [50, 100, 200],
    'min_samples_leaf': [1, 3, 5],
    'max_samples':      [0.3, 0.5, 0.7, None],
    'max_depth':        [10, 20, None]
}

svm_grid = [
    {'kernel': ['rbf'], 'gamma': [0.01, 0.1, 0.5, 1.0], 'C': [1, 10, 100]},
    {'kernel': ['linear'], 'gamma': ['scale'], 'C': [1, 10, 100]}
]

pipelines = [
    ('RF',  RandomForestClassifier(random_state=42), rf_grid),
    ('SVM', SVC(random_state=42), svm_grid),
]

In [32]:
all_results = []
best_per_model = {}

for name, model, grid in pipelines:
    print(f"\nRunning GridSearchCV for {name}")
    search = GridSearchCV(model, grid, cv=5, scoring='accuracy', return_train_score=True, n_jobs=-1)
    search.fit(X, y)
    
    cv_res = pd.DataFrame(search.cv_results_)
    cv_res['Model'] = name
    cv_res['params_str'] = cv_res['params'].astype(str)
    
    best_per_model[name] = {
        'best_params': search.best_params_,
        'best_score':  search.best_score_
    }
    print(f"Best CV accuracy: {search.best_score_:.4f}")


Running GridSearchCV for RF
Best CV accuracy: 0.8920

Running GridSearchCV for SVM
Best CV accuracy: 0.9410


In [33]:
print("=" * 55)
print("       BEST HYPERPARAMETERS PER MODEL")
print("=" * 55)
for model_name, info in best_per_model.items():
    print(f"\n{model_name}")
    print(f"   Best CV Accuracy : {info['best_score']:.4f}")
    print(f"   Best Parameters  :")
    for k, v in info['best_params'].items():
        print(f"      {k:20s} = {v}")


       BEST HYPERPARAMETERS PER MODEL (5 CLASSES)      
Model: RF
Best CV Score: 0.8920
Best Params: {'max_depth': 20, 'max_samples': None, 'min_samples_leaf': 3, 'n_estimators': 100}
-------------------------------------------------------
Model: SVM
Best CV Score: 0.9410
Best Params: {'C': 100, 'gamma': 1.0, 'kernel': 'rbf'}
